# Small Hindi → English Transformer (PyTorch, from scratch)

An encoder–decoder transformer written one function / class per job.
Run the cells **top to bottom**. Every section has: what it does, the code, and a quick sanity test.

**Pipeline**

`text file → split → tokens → ids → padded batches → masks → embeddings + positions → attention → encoder → decoder → logits → loss → backward → Adam → epochs → generate`

**Before you start:** put `hindi_english.txt` (one `hindi<TAB>english` pair per line) in the same folder as this notebook.

## 0. Imports and config

All hyperparameters live in the single `config` dict. To train on a bigger file, change `data_path`
(and set `retrain` to `True` so the old checkpoint is not reused).

In [4]:
# ---- Imports -----------------------------------------------------------------
# torch              : tensors + autograd (automatic gradients for the backward pass)
# torch.nn           : layer building blocks (Linear, Embedding, LayerNorm, Dropout ...)
# torch.nn.functional: stateless functions (softmax, relu)
# math / re / os / random : sqrt + log, regex for tokenizing, file checks, shuffling
import math
import os
import random
import re

import torch
import torch.nn as nn
import torch.nn.functional as F

In [5]:
# ---- Config: every hyperparameter and path in one place ----------------------
config = {
    # data / files
    "data_path": "hindi_english.txt",     # tab-separated: hindi <TAB> english
    "checkpoint_path": "best_model.pt",   # best weights + vocab + config are saved here
    "retrain": False,                     # False -> reuse the checkpoint if it exists
    "seed": 42,                           # same seed -> same split and same weight init

    # model size
    "d_model": 64,       # length of the vector that represents each token
    "num_heads": 4,      # attention heads (d_model must divide evenly by this)
    "d_ff": 128,         # hidden size inside the feed-forward layer
    "num_layers": 2,     # number of encoder layers AND of decoder layers
    "dropout": 0.1,      # fraction of values randomly zeroed during training
    "max_len": 100,      # longest sequence the positional encoding supports

    # training
    "lr": 5e-4,          # Adam learning rate (step size of each weight update)
    "batch_size": 64,    # sentences processed per training step
    "num_epochs": 100,    # one epoch = one full pass over the training set
}

random.seed(config["seed"])          # makes shuffling repeatable
torch.manual_seed(config["seed"])    # makes weight initialisation repeatable

## 1. Data: load and split

* **train (80%)**: the model learns from these; weights are updated on them.
* **validation (10%)**: not used for updates; checked after every epoch to spot overfitting.
* **test (10%)**: touched once at the very end for an honest final score.

In [6]:
def load_pairs_from_txt(file_path):
    """Read 'hindi<TAB>english' lines -> list of (hindi, english) tuples."""
    pairs = []
    with open(file_path, 'r', encoding='utf-8') as f:   # utf-8 is required for Hindi
        for line in f:
            line = line.strip()
            if line:                                     # skip blank lines
                hindi, english = line.split("\t")        # tab separates the two columns
                pairs.append((hindi, english))
    return pairs


def split_data_test_train(pairs, train_frac=0.8, val_frac=0.1, seed=42):
    """Shuffle reproducibly, then cut into (train, validation, test)."""
    shuffled = pairs[:]                          # copy: do not reorder the caller's list
    random.Random(seed).shuffle(shuffled)        # own Random(seed) -> same split every run

    n = len(shuffled)
    train_size = int(n * train_frac)
    val_size = int(n * val_frac)

    train_pairs = shuffled[:train_size]
    val_pairs = shuffled[train_size:train_size + val_size]
    test_pairs = shuffled[train_size + val_size:]        # whatever is left
    return train_pairs, val_pairs, test_pairs

In [7]:
# ---- Test: load + split ------------------------------------------------------
pairs_data = load_pairs_from_txt(config["data_path"])
print(len(pairs_data))                      # 10000
print(pairs_data[0])                        # a (hindi, english) tuple

train_pairs, val_pairs, test_pairs = split_data_test_train(pairs_data, seed=config["seed"])
print("Train pairs     :", len(train_pairs))        # 8000
print("Validation pairs:", len(val_pairs))          # 1000
print("Test pairs      :", len(test_pairs))         # 1000

# no sentence pair may appear in two splits
print("overlap train/val :", len(set(train_pairs) & set(val_pairs)))     # 0
print("overlap train/test:", len(set(train_pairs) & set(test_pairs)))    # 0

90000
('आदमी रात को दफ़्तर में फूल बेचता है ।', 'The man sells flowers in the office at night.')
Train pairs     : 72000
Validation pairs: 9000
Test pairs      : 9000
overlap train/val : 0
overlap train/test: 0


## 2. Tokenizing and vocabulary

A network only does maths on numbers, so text becomes **token ids**. A *vocabulary* is the lookup table `token ↔ id`.

* Word-level tokens (split on spaces) keep things simple.
* Special tokens: `<PAD>` (0) fills short sentences, `<SOS>` starts a sentence, `<EOS>` ends it, `<UNK>` replaces unseen words.
* Hindi and English each get their own vocab, built from the **train split only** (no peeking at val/test).
* `tokenize_text_line` puts a space around the danda `।`, so `है।` and `है ।` give the same tokens.

In [8]:
def tokenize_text_line(sentence):
    """'लड़की खाती है।' -> ['लड़की', 'खाती', 'है', '।']  (words split on spaces)."""
    sentence = re.sub(r"\s*।\s*", " । ", sentence)    # exactly one space on each side of the danda
    return sentence.split()


def build_vocab_from_sentences(sentences):
    """Return (stoi, itos): stoi = token -> id (dict), itos = id -> token (list)."""
    special_tokens = ['<PAD>', '<SOS>', '<EOS>', '<UNK>']          # ids 0, 1, 2, 3
    all_tokens = sorted({tok for s in sentences for tok in tokenize_text_line(s)})   # unique + fixed order
    itos = special_tokens + all_tokens
    stoi = {tok: i for i, tok in enumerate(itos)}
    return stoi, itos


def encode_sentence(sentence, stoi):
    """Text -> [<SOS>, id, id, ..., <EOS>]. Unknown words become <UNK>."""
    tokens = tokenize_text_line(sentence)
    encoded = [stoi.get(token, stoi['<UNK>']) for token in tokens]
    return [stoi['<SOS>']] + encoded + [stoi['<EOS>']]


def decode_sequence(encoded_ids, itos):
    """ids -> text. <PAD>/<SOS>/<EOS> are dropped."""
    skip = ['<PAD>', '<SOS>', '<EOS>']
    words = [itos[i] for i in encoded_ids if i < len(itos) and itos[i] not in skip]
    return ' '.join(words)

In [9]:
# ---- Test: vocab + encode/decode round trip ----------------------------------
src_stoi, src_itos = build_vocab_from_sentences([p[0] for p in train_pairs])   # Hindi vocab
tgt_stoi, tgt_itos = build_vocab_from_sentences([p[1] for p in train_pairs])   # English vocab
print(len(src_stoi), len(tgt_stoi))                    # 77 98

ids = encode_sentence(train_pairs[0][1], tgt_stoi)
print(ids)                                             # starts with 1 (<SOS>), ends with 2 (<EOS>)
print(decode_sequence(ids, tgt_itos))                  # the original English sentence

# the danda fix: with or without a space gives the same tokens
print(tokenize_text_line("लड़की रोज़ आम खाती है।"))
print(tokenize_text_line("लड़की रोज़ आम खाती है ।"))

262 286
[1, 34, 113, 76, 125, 137, 252, 128, 110, 93, 2]
The farmer buys fruits in the garden every day.
['लड़की', 'रोज़', 'आम', 'खाती', 'है', '।']
['लड़की', 'रोज़', 'आम', 'खाती', 'है', '।']


## 3. Batching: padding and the decoder shift

Training works on many sentences at once as one rectangular tensor `[batch, length]`, so short sentences are padded with `<PAD>` (0).

**Teacher forcing** - the decoder gets the correct English sentence shifted by one:

```
tgt_in  = <SOS> They sell milk        (what the decoder reads)
tgt_out = They  sell milk <EOS>       (what it must predict at each position)
```

Slice **before** padding, so `<EOS>` never ends up inside `tgt_in`.

In [10]:
def pad_sequences(batch_ids, pad_token_id):
    """List of id-lists (different lengths) -> LongTensor [batch, longest]."""
    max_length = max(len(ids) for ids in batch_ids)             # longest sentence in this batch
    padded = []
    for ids in batch_ids:
        n_pad = max_length - len(ids)                           # how many pads this one needs
        padded.append(ids + [pad_token_id] * n_pad)
    return torch.tensor(padded, dtype=torch.long)               # ids must be integers (embedding lookup)


def make_batch(pairs, src_stoi, tgt_stoi):
    """pairs -> (src, tgt_in, tgt_out) padded tensors."""
    src_encoded = [encode_sentence(p[0], src_stoi) for p in pairs]     # Hindi ids
    tgt_encoded = [encode_sentence(p[1], tgt_stoi) for p in pairs]     # English ids

    tgt_in_encoded = [ids[:-1] for ids in tgt_encoded]     # <SOS> w1 w2 ...  (drop <EOS>)
    tgt_out_encoded = [ids[1:] for ids in tgt_encoded]     # w1 w2 ... <EOS>  (drop <SOS>)

    src_padded = pad_sequences(src_encoded, src_stoi['<PAD>'])
    tgt_in_padded = pad_sequences(tgt_in_encoded, tgt_stoi['<PAD>'])
    tgt_out_padded = pad_sequences(tgt_out_encoded, tgt_stoi['<PAD>'])
    return src_padded, tgt_in_padded, tgt_out_padded


def get_batches(pairs, batch_size, src_stoi, tgt_stoi, shuffle=True):
    """Generator: yields one (src, tgt_in, tgt_out) batch at a time."""
    pairs = pairs[:]                       # copy so shuffling does not reorder the caller's list
    if shuffle:
        random.shuffle(pairs)              # new order every epoch (training); off for val/test
    for i in range(0, len(pairs), batch_size):
        batch_pairs = pairs[i:i + batch_size]
        yield make_batch(batch_pairs, src_stoi, tgt_stoi)

In [11]:
# ---- Test: padding, shift, shapes --------------------------------------------
print(pad_sequences([[1, 9, 78, 2], [1, 9, 78, 57, 14, 2], [1, 9, 2]], 0))   # 3 rows, 6 columns

src, tgt_in, tgt_out = next(get_batches(train_pairs, 64, src_stoi, tgt_stoi))
print(src.shape, tgt_in.shape, tgt_out.shape)        # first dim 64; tgt_in and tgt_out same shape
print(tgt_in[0])                                     # starts with 1 (<SOS>)
print(tgt_out[0])                                    # same ids moved one place left, 2 (<EOS>) after the last word

tensor([[ 1,  9, 78,  2,  0,  0],
        [ 1,  9, 78, 57, 14,  2],
        [ 1,  9,  2,  0,  0,  0]])
torch.Size([64, 21]) torch.Size([64, 14]) torch.Size([64, 14])
tensor([  1,  34, 241,  72,  70, 137, 252, 184, 110, 277,   0,   0,   0,   0])
tensor([ 34, 241,  72,  70, 137, 252, 184, 110, 277,   2,   0,   0,   0,   0])


## 4. Masks

A mask is a True/False grid saying which positions attention may look at (**True = allowed**).

* **Padding mask** - hides `<PAD>` filler. Shape `[B, 1, 1, L]`.
* **Causal mask** - decoder only; a word may not see later words (a lower-triangular matrix). Shape `[1, 1, T, T]`.
* **Target mask** = padding AND causal, so the decoder sees only real, earlier words.

The extra size-1 dimensions broadcast over the attention heads (and the batch for the causal mask).

In [12]:
def make_pad_mask(encoded_ids, pad_token_id):
    """[B, L] ids -> [B, 1, 1, L] bool. True = real token, False = padding."""
    mask = (encoded_ids != pad_token_id)
    return mask.unsqueeze(1).unsqueeze(2)


def make_causal_mask(encoded_ids):
    """[B, T] ids -> [1, 1, T, T] bool lower triangle (row i may see columns 0..i)."""
    seq_len = encoded_ids.size(1)
    causal_mask = torch.tril(torch.ones((seq_len, seq_len), dtype=torch.bool))
    return causal_mask.unsqueeze(0).unsqueeze(1)          # add batch and head dimensions


def make_tgt_mask(encoded_ids, pad_token_id):
    """Decoder mask: not padding AND not in the future -> [B, 1, T, T] bool."""
    pad_mask = make_pad_mask(encoded_ids, pad_token_id)       # [B, 1, 1, T]
    causal_mask = make_causal_mask(encoded_ids)               # [1, 1, T, T]
    return pad_mask & causal_mask                             # broadcasts to [B, 1, T, T]

In [13]:
# ---- Test: masks -------------------------------------------------------------
src_mask = make_pad_mask(src, src_stoi['<PAD>'])
print(src_mask.shape, src_mask.dtype)                    # [64, 1, 1, L]   torch.bool

print(make_causal_mask(tgt_in[:, :4])[0, 0].int())       # 4x4 lower triangle

tgt_mask = make_tgt_mask(tgt_in, tgt_stoi['<PAD>'])
print(tgt_mask.shape)                                    # [64, 1, L, L]
print(tgt_mask[0, 0].int())                              # triangle that stops at the last real token

torch.Size([64, 1, 1, 21]) torch.bool
tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]], dtype=torch.int32)
torch.Size([64, 1, 14, 14])
tensor([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]], dtype=torch.int32)


## 5. Embedding and positional encoding

* **Embedding** - a learnable table `[vocab, d_model]`; token id -> its vector. Scaled by `sqrt(d_model)` as in the paper.
* **Positional encoding** - attention has no sense of word order, so a fixed sin/cos pattern for each position is added:
  `PE[pos, 2i] = sin(pos / 10000^(2i/d))`, `PE[pos, 2i+1] = cos(...)`. It is a *buffer*: saved with the model, never trained.

In [14]:
class TokenEmbedding(nn.Module):
    """ids [B, L] -> vectors [B, L, d_model]."""
    def __init__(self, vocab_size, d_model, pad_id):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)   # <PAD> row stays zeros
        self.d_model = d_model

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)     # scale so positions do not drown it out


class PositionalEncoding(nn.Module):
    """Adds a fixed position-dependent vector to every token vector."""
    def __init__(self, d_model, max_len, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)                                    # table [max_len, d_model]
        position = torch.arange(max_len).unsqueeze(1)                         # [max_len, 1]
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)                          # even columns: sin
        pe[:, 1::2] = torch.cos(position * div_term)                          # odd columns : cos
        pe = pe.unsqueeze(0)                                                  # [1, max_len, d_model]
        self.register_buffer("pe", pe)           # part of the model, but NOT a trainable weight

    def forward(self, x):                        # x: [B, L, d_model]
        x = x + self.pe[:, :x.size(1)]           # add the first L rows of the table
        return self.dropout(x)

In [15]:
# ---- Test: embedding + positions ---------------------------------------------
emb = TokenEmbedding(len(src_stoi), config["d_model"], src_stoi['<PAD>'])
pos_enc = PositionalEncoding(config["d_model"], config["max_len"], config["dropout"])

x = emb(src)
print(x.shape)                         # [64, L, 64]
print(pos_enc(x).shape)                # same shape: positions are added, not appended
print(pos_enc.pe[0, 0, :4])            # tensor([0., 1., 0., 1.])  position 0 = sin(0), cos(0), ...
print(pos_enc.pe[0, 1, :4])            # position 1 differs from position 0

torch.Size([64, 21, 64])
torch.Size([64, 21, 64])
tensor([0., 1., 0., 1.])
tensor([0.8415, 0.5403, 0.6816, 0.7318])


## 6. Attention

**Scaled dot-product attention** - each word makes a *query*, every word offers a *key* and a *value*.
Match score = query · key; softmax turns scores into weights; the output is the weighted average of the values:

`Attention(Q, K, V) = softmax(Q·Kᵀ / sqrt(d_k)) · V`

Masked positions get a huge negative score before the softmax, so their weight becomes ~0.

**Multi-head attention** splits `d_model` into several heads that each attend differently, then joins them.
The same class does *self*-attention (`q = k = v`) and *cross*-attention (`q` from the decoder, `k = v` from the encoder).

In [16]:
def scaled_dot_product_attention(query, key, value, mask=None):
    """q, k, v: [B, H, L, d_k]  ->  (output [B, H, Lq, d_k], weights [B, H, Lq, Lk])."""
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)    # compare every query with every key

    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)     # blocked positions -> softmax weight ~0

    attention_weights = F.softmax(scores, dim=-1)        # each row sums to 1
    output = torch.matmul(attention_weights, value)      # weighted average of the values
    return output, attention_weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.num_heads = num_heads
        self.d_k = d_model // num_heads                   # size of each head's slice
        self.W_q = nn.Linear(d_model, d_model)            # learnable projections for Q, K, V ...
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)            # ... and for mixing the joined heads

    def split_heads(self, x):                             # [B, L, d_model] -> [B, H, L, d_k]
        B, L, _ = x.size()
        return x.view(B, L, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        B, Lq, _ = query.size()
        q = self.split_heads(self.W_q(query))
        k = self.split_heads(self.W_k(key))
        v = self.split_heads(self.W_v(value))
        out, weights = scaled_dot_product_attention(q, k, v, mask)
        out = out.transpose(1, 2).contiguous().view(B, Lq, self.num_heads * self.d_k)   # join the heads
        return self.W_o(out)

In [17]:
# ---- Test: attention ---------------------------------------------------------
q = torch.randn(2, 4, 5, 16); k = torch.randn(2, 4, 5, 16); v = torch.randn(2, 4, 5, 16)

out, w = scaled_dot_product_attention(q, k, v)
print(out.shape, w.shape)                      # [2,4,5,16]  [2,4,5,5]
print(w.sum(dim=-1)[0, 0])                     # all ones

causal = make_causal_mask(torch.zeros(2, 5))
out, w = scaled_dot_product_attention(q, k, v, causal)
print(w[0, 0])                                 # zeros above the diagonal; first row is [1, 0, 0, 0, 0]

mha = MultiHeadAttention(config["d_model"], config["num_heads"])
xx = torch.randn(2, 5, 64)
print(mha(xx, xx, xx).shape)                                       # self-attention  [2, 5, 64]
print(mha(torch.randn(2, 3, 64), torch.randn(2, 7, 64), torch.randn(2, 7, 64)).shape)   # cross-attention [2, 3, 64]
print(sum(p.numel() for p in mha.parameters()))                    # 16640

torch.Size([2, 4, 5, 16]) torch.Size([2, 4, 5, 5])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6830, 0.3170, 0.0000, 0.0000, 0.0000],
        [0.3534, 0.3184, 0.3282, 0.0000, 0.0000],
        [0.1581, 0.1987, 0.1749, 0.4683, 0.0000],
        [0.3362, 0.2521, 0.1251, 0.2676, 0.0191]])
torch.Size([2, 5, 64])
torch.Size([2, 3, 64])
16640


## 7. Feed-forward and the encoder layer

* **Feed-forward** - works on each word's vector on its own: `Linear(64→128) → ReLU → Dropout → Linear(128→64)`.
* **Residual connection** `x + sublayer(x)` keeps the original information and lets gradients flow back through deep stacks.
* **LayerNorm** rescales each word's 64 numbers to mean 0 / std 1, which keeps training stable.

Encoder layer = `LayerNorm(x + Dropout(SelfAttention(x)))` then `LayerNorm(x + Dropout(FeedForward(x)))`.

In [18]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)      # expand 64 -> 128
        self.linear2 = nn.Linear(d_ff, d_model)      # shrink back 128 -> 64
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))     # linear1 -> ReLU -> dropout -> linear2


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, src_mask)))    # block 1: words look at each other
        x = self.norm2(x + self.dropout(self.ff(x)))                           # block 2: per-word processing
        return x

In [19]:
# ---- Test: feed-forward + encoder layer --------------------------------------
ff = FeedForward(config["d_model"], config["d_ff"], config["dropout"])
print(sum(p.numel() for p in ff.parameters()))           # 16576

enc_layer = EncoderLayer(config["d_model"], config["num_heads"], config["d_ff"], config["dropout"])
print(sum(p.numel() for p in enc_layer.parameters()))    # 33472

x = pos_enc(emb(src))
out = enc_layer(x, src_mask)
print(out.shape)                                         # unchanged: [64, L, 64]
print(out.mean(dim=-1)[0, :3])                           # ~0   (LayerNorm)
print(out.std(dim=-1, unbiased=False)[0, :3])            # ~1

16576
33472
torch.Size([64, 21, 64])
tensor([-3.7253e-09,  0.0000e+00, -1.4901e-08], grad_fn=<SliceBackward0>)
tensor([1.0000, 1.0000, 1.0000], grad_fn=<SliceBackward0>)


## 8. Encoder, decoder and the full Transformer

* **Encoder** - embedding → positions → N encoder layers. Output: one rich vector per Hindi word.
* **Decoder layer** - 3 blocks: masked self-attention (`tgt_mask`), **cross-attention** (query = English side, key/value = encoder output, `src_mask`), feed-forward.
* **Transformer** - encoder + decoder + a final `Linear(d_model → English vocab)` that gives **logits** (raw scores per word).

The model returns *logits*, not probabilities: `CrossEntropyLoss` applies the softmax itself.
`nn.ModuleList` is required so PyTorch registers (and trains) every layer.

In [20]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout, max_len, pad_id):
        super().__init__()
        self.embedding = TokenEmbedding(vocab_size, d_model, pad_id)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

    def forward(self, src, src_mask):
        x = self.pos_enc(self.embedding(src))       # ids -> vectors + positions
        for layer in self.layers:                   # pass through each encoder layer in turn
            x = layer(x, src_mask)
        return x                                    # [B, S, d_model]


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)     # English words look at earlier English words
        self.cross_attn = MultiHeadAttention(d_model, num_heads)    # English words look at the Hindi sentence
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, tgt_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))                 # 1. masked self-attention
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out, enc_out, src_mask)))    # 2. cross-attention
        x = self.norm3(x + self.dropout(self.ff(x)))                                        # 3. feed-forward
        return x


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout, max_len, pad_id):
        super().__init__()
        self.embedding = TokenEmbedding(vocab_size, d_model, pad_id)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

    def forward(self, tgt_in, enc_out, src_mask, tgt_mask):
        x = self.pos_enc(self.embedding(tgt_in))
        for layer in self.layers:
            x = layer(x, enc_out, src_mask, tgt_mask)
        return x                                    # [B, T, d_model]


class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model, num_heads, d_ff, num_layers,
                 dropout, max_len, src_pad_id, tgt_pad_id):
        super().__init__()
        self.src_pad_id = src_pad_id
        self.tgt_pad_id = tgt_pad_id
        self.encoder = Encoder(src_vocab, d_model, num_heads, d_ff, num_layers, dropout, max_len, src_pad_id)
        self.decoder = Decoder(tgt_vocab, d_model, num_heads, d_ff, num_layers, dropout, max_len, tgt_pad_id)
        self.output_layer = nn.Linear(d_model, tgt_vocab)       # 64 numbers -> one score per English word

    def forward(self, src, tgt_in):
        src_mask = make_pad_mask(src, self.src_pad_id)          # masks are built here, from the pad ids
        tgt_mask = make_tgt_mask(tgt_in, self.tgt_pad_id)
        enc_out = self.encoder(src, src_mask)                   # understand the Hindi sentence
        dec_out = self.decoder(tgt_in, enc_out, src_mask, tgt_mask)
        return self.output_layer(dec_out)                       # logits [B, T, tgt_vocab]


def build_model(config, src_stoi, tgt_stoi):
    """One place that creates the network (used by training and by checkpoint loading)."""
    return Transformer(len(src_stoi), len(tgt_stoi), config["d_model"], config["num_heads"],
                       config["d_ff"], config["num_layers"], config["dropout"], config["max_len"],
                       src_stoi['<PAD>'], tgt_stoi['<PAD>'])

In [21]:
# ---- Test: the full forward pass up to softmax --------------------------------
test_model = build_model(config, src_stoi, tgt_stoi)
logits = test_model(src, tgt_in)
print(logits.shape)                                          # [64, L, 98]
print(sum(p.numel() for p in test_model.parameters()))       # 184994

probs = torch.softmax(logits, dim=-1)                        # scores -> probabilities
print(probs.sum(dim=-1)[0, :3])                              # all ones
print(probs[0, 0].max().item())                              # small (~0.01-0.03): untrained = near-uniform

# causal check: changing a FUTURE word must not change earlier predictions
test_model.eval()                                            # dropout off -> repeatable
with torch.no_grad():
    l1 = test_model(src, tgt_in)
    tgt2 = tgt_in.clone()
    tgt2[:, 5] = 7                                           # change the word at position 5
    l2 = test_model(src, tgt2)
print(torch.allclose(l1[:, :5], l2[:, :5], atol=1e-5))       # True : positions 0-4 cannot see position 5
print(torch.allclose(l1[:, 5:], l2[:, 5:], atol=1e-5))       # False: positions 5+ do change

torch.Size([64, 14, 286])
221086
tensor([1.0000, 1.0000, 1.0000], grad_fn=<SliceBackward0>)
0.010922177694737911
True
False


## 9. Loss, backward pass and Adam: one training step

```
forward -> loss -> zero_grad -> backward -> clip -> optimizer.step
```

* **Cross-entropy loss** = `-log(probability given to the correct word)`, averaged over positions. `ignore_index` skips `<PAD>` positions.
  An untrained model scores about `ln(vocab) ≈ ln(98) ≈ 4.6`.
* **`loss.backward()`** computes a *gradient* (how the loss changes if the weight changes) for every weight.
* **`zero_grad()`** first, otherwise gradients pile up across batches.
* **Gradient clipping** caps huge gradients so one bad batch cannot wreck training.
* **Adam** gives every weight its own step size using running averages of its gradient.

In [22]:
def compute_loss(logits, tgt_out, criterion):
    """logits [B, T, V] and targets [B, T] -> one number. Flatten batch+time for CrossEntropyLoss."""
    V = logits.size(-1)
    return criterion(logits.reshape(-1, V), tgt_out.reshape(-1))


def training_step(model, batch, criterion, optimizer):
    """One batch: forward, loss, backward, update. Returns the loss as a float."""
    src, tgt_in, tgt_out = batch
    model.train()                                                  # dropout ON
    logits = model(src, tgt_in)                                    # 1. forward pass
    loss = compute_loss(logits, tgt_out, criterion)                # 2. how wrong were we?
    optimizer.zero_grad()                                          # 3. clear old gradients
    loss.backward()                                                # 4. backward pass: gradient for every weight
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)     #    cap very large gradients
    optimizer.step()                                               # 5. Adam updates the weights
    return loss.item()                                             # plain number for logging

In [23]:
# ---- Test: first loss, then overfit ONE batch (proves backward + Adam work) ---
tmp_model = build_model(config, src_stoi, tgt_stoi)                # throw-away model for this test
tmp_criterion = nn.CrossEntropyLoss(ignore_index=tgt_stoi['<PAD>'])
tmp_optimizer = torch.optim.Adam(tmp_model.parameters(), lr=config["lr"], betas=(0.9, 0.98), eps=1e-9)

one_batch = make_batch(train_pairs[:64], src_stoi, tgt_stoi)
for step in range(100):
    loss = training_step(tmp_model, one_batch, tmp_criterion, tmp_optimizer)
    if step % 20 == 0:
        print(step, round(loss, 4))        # starts near 4.6 (= ln 98), then falls steadily

0 5.8596
20 4.3991
40 3.5062
60 2.83
80 2.2565


## 10. Epochs, validation and checkpoints

* **Step** = one batch. **Epoch** = enough steps to see the whole training set once.
* `train_one_epoch` learns; `evaluate` only measures (`model.eval()`, `torch.no_grad()`, no weight updates).
* Read the two numbers together: both falling = learning. Train falling while validation rises = **overfitting**.
* The best-validation model is saved as a **checkpoint** that also stores the vocab and config, so generation works later without retraining.

In [24]:
def train_one_epoch(model, pairs, batch_size, src_stoi, tgt_stoi, criterion, optimizer):
    """One full pass over `pairs`, learning on every batch. Returns the average loss."""
    losses = []
    for batch in get_batches(pairs, batch_size, src_stoi, tgt_stoi, shuffle=True):
        losses.append(training_step(model, batch, criterion, optimizer))
    return sum(losses) / len(losses)


def evaluate(model, pairs, batch_size, src_stoi, tgt_stoi, criterion):
    """Average loss on `pairs` WITHOUT learning (used for validation and test)."""
    model.eval()                                   # dropout OFF -> repeatable measurement
    losses = []
    with torch.no_grad():                          # no gradient bookkeeping: faster, less memory
        for src, tgt_in, tgt_out in get_batches(pairs, batch_size, src_stoi, tgt_stoi, shuffle=False):
            logits = model(src, tgt_in)
            losses.append(compute_loss(logits, tgt_out, criterion).item())
    return sum(losses) / len(losses)


def save_checkpoint(path, model, src_itos, tgt_itos, config):
    """Save weights + vocab lists + config in one file (stoi is rebuilt from itos on load)."""
    torch.save({"model_state": model.state_dict(),
                "src_itos": src_itos, "tgt_itos": tgt_itos, "config": config}, path)


def load_checkpoint(path):
    """Rebuild the model exactly as it was saved. Returns (model, src_stoi, tgt_stoi, tgt_itos)."""
    ckpt = torch.load(path)
    if "model_state" not in ckpt:
        raise ValueError(f"{path} is an old weights-only file. Delete it, or set config['retrain'] = True.")
    src_itos, tgt_itos, saved_config = ckpt["src_itos"], ckpt["tgt_itos"], ckpt["config"]
    src_stoi = {tok: i for i, tok in enumerate(src_itos)}
    tgt_stoi = {tok: i for i, tok in enumerate(tgt_itos)}
    model = build_model(saved_config, src_stoi, tgt_stoi)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model, src_stoi, tgt_stoi, tgt_itos


def train_model(train_pairs, val_pairs, src_vocab, tgt_vocab, config):
    """Full training run. Saves the best-validation model to config['checkpoint_path']."""
    src_stoi, src_itos = src_vocab
    tgt_stoi, tgt_itos = tgt_vocab

    model = build_model(config, src_stoi, tgt_stoi)                          # fresh weights every call
    criterion = nn.CrossEntropyLoss(ignore_index=tgt_stoi['<PAD>'])          # skip <PAD> positions
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], betas=(0.9, 0.98), eps=1e-9)

    best_val = float("inf")
    for epoch in range(1, config["num_epochs"] + 1):
        train_loss = train_one_epoch(model, train_pairs, config["batch_size"], src_stoi, tgt_stoi, criterion, optimizer)
        val_loss = evaluate(model, val_pairs, config["batch_size"], src_stoi, tgt_stoi, criterion)
        print(f"epoch {epoch:2d} | train {train_loss:.4f} | val {val_loss:.4f}")
        if val_loss < best_val:                                              # keep the best model so far
            best_val = val_loss
            save_checkpoint(config["checkpoint_path"], model, src_itos, tgt_itos, config)
    return model

## 11. Generation and final test

**Autoregressive greedy decoding** - build the English sentence one word at a time: run the decoder on the words so far,
take the scores of the *last* position only, pick the highest, append it, repeat until `<EOS>`.
The encoder runs once; the Hindi side never changes.

**Exact match** - the share of sentences translated word for word like the reference (a stricter score than loss).

In [25]:
def translate(model, sentence, src_stoi, tgt_stoi, tgt_itos, max_len=20):
    """Hindi sentence (str) -> English sentence (str), greedy decoding."""
    model.eval()
    pe_len = model.encoder.pos_enc.pe.size(1)                       # longest length the model supports
    src_ids = encode_sentence(sentence, src_stoi)
    if len(src_ids) > pe_len:
        raise ValueError(f"sentence has {len(src_ids)} tokens; model supports {pe_len}. Raise config['max_len'].")
    max_len = min(max_len, pe_len - 1)

    with torch.no_grad():
        src = torch.tensor([src_ids])                                # [1, S]
        src_mask = make_pad_mask(src, src_stoi['<PAD>'])
        enc_out = model.encoder(src, src_mask)                       # encoder runs ONCE

        tgt = torch.tensor([[tgt_stoi['<SOS>']]])                    # start with just <SOS>
        for _ in range(max_len):
            tgt_mask = make_tgt_mask(tgt, tgt_stoi['<PAD>'])         # rebuilt: tgt grows each step
            dec_out = model.decoder(tgt, enc_out, src_mask, tgt_mask)
            next_logits = model.output_layer(dec_out[:, -1])         # scores for the NEXT word only
            next_id = torch.argmax(next_logits, dim=-1)              # greedy: most likely word
            tgt = torch.cat([tgt, next_id.view(1, 1)], dim=1)        # feed it back in
            if next_id.item() == tgt_stoi['<EOS>']:                  # model says "sentence finished"
                break
    return decode_sequence(tgt[0].tolist(), tgt_itos)


def exact_match_accuracy(model, pairs, src_stoi, tgt_stoi, tgt_itos):
    """Share of pairs translated exactly like the reference. Returns (accuracy, mistakes)."""
    correct = 0
    mistakes = []                                                    # (hindi, reference, model output)
    for hi, en in pairs:
        out = translate(model, hi, src_stoi, tgt_stoi, tgt_itos)
        ref = " ".join(tokenize_text_line(en))                       # same tokenizing as the output
        if out == ref:
            correct += 1
        else:
            mistakes.append((hi, ref, out))
    return correct / len(pairs), mistakes

## 12. `main()`: train (if needed) → test → return the model

`main()` **never waits for keyboard input**, so it always finishes.
It trains only when there is no checkpoint (or `config["retrain"]` is `True`), prints the test loss and exact-match score,
shows a few translations, and returns the loaded model so the next cell can use it.

In [26]:
def main():
    random.seed(config["seed"])
    torch.manual_seed(config["seed"])

    # 1. data -> splits -> vocab (vocab from the TRAIN split only)
    pairs = load_pairs_from_txt(config["data_path"])
    train_pairs, val_pairs, test_pairs = split_data_test_train(pairs, seed=config["seed"])
    src_vocab = build_vocab_from_sentences([p[0] for p in train_pairs])
    tgt_vocab = build_vocab_from_sentences([p[1] for p in train_pairs])
    print(f"data: {len(train_pairs)} train / {len(val_pairs)} val / {len(test_pairs)} test | "
          f"vocab: {len(src_vocab[0])} Hindi, {len(tgt_vocab[0])} English")

    # 2. train only when needed
    if config["retrain"] or not os.path.exists(config["checkpoint_path"]):
        train_model(train_pairs, val_pairs, src_vocab, tgt_vocab, config)
    else:
        print(f"found {config['checkpoint_path']} - skipping training (set config['retrain'] = True to retrain)")

    # 3. reload the BEST checkpoint (not necessarily the last epoch)
    model, src_stoi, tgt_stoi, tgt_itos = load_checkpoint(config["checkpoint_path"])

    # 4. final score on the untouched test split
    criterion = nn.CrossEntropyLoss(ignore_index=tgt_stoi['<PAD>'])
    test_loss = evaluate(model, test_pairs, config["batch_size"], src_stoi, tgt_stoi, criterion)
    acc, mistakes = exact_match_accuracy(model, test_pairs, src_stoi, tgt_stoi, tgt_itos)
    print(f"test loss: {test_loss:.4f} | exact match: {acc:.2%} ({len(test_pairs) - len(mistakes)}/{len(test_pairs)})")
    for hi, ref, out in mistakes[:5]:                                # look at failures, if any
        print("  HI :", hi, "\n  REF:", ref, "\n  OUT:", out)

    # 5. a few example translations
    for hi, en in test_pairs[:3]:
        print("HI :", hi, "\nREF:", en, "\nOUT:", translate(model, hi, src_stoi, tgt_stoi, tgt_itos), "\n")

    return model, src_stoi, tgt_stoi, tgt_itos

In [27]:
# ---- Run the whole pipeline (finishes on its own; no input box) --------------
model, src_stoi, tgt_stoi, tgt_itos = main()

data: 72000 train / 9000 val / 9000 test | vocab: 262 Hindi, 286 English
found best_model.pt - skipping training (set config['retrain'] = True to retrain)
test loss: 6.5850 | exact match: 10.63% (957/9000)
  HI : आदमी रविवार को बाज़ार में बैग चुनता है । 
  REF: The man chooses a bag at the market on Sunday. 
  OUT: The man sees a movie in the garden on Sunday.
  HI : आदित्य, क्या आप कृपया मुझे सप्ताहांत की तस्वीरें अभी भेज सकते हैं? 
  REF: Aditya, could you please send me the weekend photos now? 
  OUT: The woman drinks tea. drinks tea. drinks tea.
  HI : शिक्षक हर दिन बगीचे में किताब चुनता है । 
  REF: The teacher chooses a book in the garden every day. 
  OUT: The teacher buys a book in the garden every evening.
  HI : छात्र हर दिन बगीचे में चावल लाता है । 
  REF: The student brings rice in the garden every day. 
  OUT: The students eat rice in the garden every evening.
  HI : अर्जुन, क्या आप कृपया मुझे कार्यालय का नक्शा बैठक से पहले भेज सकते हैं? 
  REF: Arjun, could you please sen

## 13. Try your own sentences

Edit the strings and re-run. Rules for good queries:
* Use only words from the training data (about 77 Hindi words); any other word becomes `<UNK>` and the output is unreliable.
* The danda may be attached (`है।`) or spaced (`है ।`); both work.
* Recombinations of known words that were never seen together are a fair test that the model learned the grammar.

In [28]:
def ask(sentence):
    """Translate one Hindi sentence with the trained model and print both."""
    print("HI :", sentence)
    print("EN :", translate(model, sentence, src_stoi, tgt_stoi, tgt_itos), "\n")


ask("लड़की रोज़ आम खाती है।")                   # danda attached to the last word
ask("वे हर शाम बगीचे में दूध पीते हैं ।")        # a recombination of known words
ask("शिक्षक रविवार को स्कूल में कहानी लिखता है ।")

HI : लड़की रोज़ आम खाती है।
EN : The girl eats a mango every day. 

HI : वे हर शाम बगीचे में दूध पीते हैं ।
EN : They drink milk in the garden every evening. 

HI : शिक्षक रविवार को स्कूल में कहानी लिखता है ।
EN : The teacher writes a story at school on Sunday. 



In [29]:
# ---- Optional: interactive loop for a TERMINAL / .py script only -------------
# input() waits for the keyboard, which is why it is NOT part of main().
# In a notebook, use ask("...") above instead. To use it, call chat() yourself.
"""def chat():
    while True:
        q = input("Hindi> ").strip()
        if q in ("", "quit"):
            break
        ask(q)
"""

'def chat():\n    while True:\n        q = input("Hindi> ").strip()\n        if q in ("", "quit"):\n            break\n        ask(q)\n'

In [31]:
ask("केशव स्कूल नहीं जाता है।")

HI : केशव स्कूल नहीं जाता है।
EN : The man sees a poem. school. 

